<a href="https://colab.research.google.com/github/anra8571/INFO5871FinalProject/blob/main/Assignment6_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 6 Improved Evaluation and Finetuning (40 Points)

This assignment will build on assignment 4 in which we loaded and inspected the SQuAD dataset, loaded and used BERT-mini, and observed BERT-mini's out-of-the-box performance on the SQuAD dataset. We didn't get very good performance on out-of-the-box BERT-mini.  There were many problems and challenges:

 * the evaluation criteria was very strict requiring exact string matching.
 * BERT-mini is not finetuned on the Q/A task
 * BERT-mini is very small and might not be large enough to perform well at this task.  

There isn't much we can do about the size of BERT-mini except to use a larger model.  If you have time and computational resources, then you might consider doing the exercises in this notebook with a larger model to see how much better it works.  For the purposes of this homework, we are going to focus on improving the evaluation criteria and finetuning BERT-mini.

# Evaluation

Let's pick up where we left off by running an example from SQuAD through BERT-mini

In [2]:
!pip install tokenizers==0.20

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 17.6 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.20.3
    Uninstalling tokenizers-0.20.3:
      Successfully uninstalled tokenizers-0.20.3


In [3]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [4]:
# run this block to import the necessary libraries
from transformers import BertTokenizerFast, BertForQuestionAnswering, Trainer, TrainingArguments
from datasets import load_dataset
import torch
import random
from collections import Counter
import statistics
import json
import numpy as np
import re

In [5]:
# load SQuAD
squad = load_dataset("squad")
print(squad)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


## Setting a seed

Several students mentioned that they got inconsisten result from the model between one run and the next.  A quick search suggests that this inconsistent behavior can be mitigated by setting a seed value for both numpy and torch.  Let's do that here so that we have a chance of seeing the same behavior.  If you do not see exactly what I see in the examples below, don't worry too much.

In [6]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"Seed set to: {seed}")

set_seed(155)

Seed set to: 155


In [7]:
# let's load the BERT-mini model and its tokenzier
# Note - we need to use BertTokenizerFast for reasons explained in the first TODO programming block
tokenizer = BertTokenizerFast.from_pretrained('prajjwal1/bert-mini')
model = BertForQuestionAnswering.from_pretrained('prajjwal1/bert-mini')

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/45.1M [00:00<?, ?B/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at prajjwal1/bert-mini and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Warning Messages
When I load BERT-mini I sometimes get the following "FutureWarning":

```
/opt/anaconda3/envs/csci5832/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
```

I consistently get the following warning:

```
Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at prajjwal1/bert-mini and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
```

If you read this message carefully, it makes sense that Q/A related weights are not initialized from the "model checkpoint".  That is, the model did not have the weights and so they were initialized with "new values" - i.e. random, untrained values. By the time we get done with this assignment, we will have a model that loads without giving us this warning.  

In [8]:
# here is my implementation of `answer_question` from Assignment 4
def answer_question(question, context, model, tokenizer):
    inputs = tokenizer(question, context, return_tensors='pt', truncation=True, max_length=512)

    input_ids = inputs["input_ids"].tolist()[0]
    token_type_ids = inputs['token_type_ids']
    context_start_index = (token_type_ids == 1).nonzero(as_tuple=True)[1].min().item()

    with torch.no_grad():
        outputs = model(**inputs)
        start_scores = outputs.start_logits
        end_scores = outputs.end_logits

    start_index = torch.argmax(start_scores[0][context_start_index:]) + context_start_index
    end_index = torch.argmax(end_scores[0][start_index:]) + start_index

    answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(input_ids[start_index:end_index]))

    return answer

## Testing Random Examples

In [9]:
# test a random example - rerun this several times to get some sense of how the model is behaving out of the box
index = random.randint(0, len(squad['train']) - 1)
example = squad['train'][index]
question = example['question']
context = example['context']
print(f"example[{index}]: question = '{question}'\ncontext = '{context}'")
print(f"expected answer = {example['answers']['text'][0]}")
actual_answer = answer_question(question, context, model, tokenizer)
print(f"actual answer = {actual_answer}")

example[76137]: question = 'In what years did the Washington University men's basketball teams win division championships?'
context = 'Washington University's sports teams are called the Bears. They are members of the National Collegiate Athletic Association and participate in the University Athletic Association at the Division III level. The Bears have won 19 NCAA Division III Championships— one in women's cross country (2011), one in men's tennis (2008), two in men's basketball (2008, 2009), five in women's basketball (1998–2001, 2010), and ten in women's volleyball (1989, 1991–1996, 2003, 2007, 2009) – and 144 UAA titles in 15 different sports. The Athletic Department is headed by John Schael who has served as director of athletics since 1978. The 2000 Division III Central Region winner of the National Association of Collegiate Directors of Athletics/Continental Airlines Athletics Director of the Year award, Schael has helped orchestrate the Bears athletics transformation into one o

## Vogue Ok!
When I run the above code to observe BERT-mini attempting to answer questions from random training examples, it mostly seems to be falling down pretty hard - i.e. it's not really very close to getting the correct answer.  However, if you look at many examples, occasionally you will see one where the answer was pretty close - but not an exact string match with the expected answer.  For example, try training example 25,843:

In [10]:
# with seed = 155
index = 25_843
example = squad['train'][index]
question = example['question']
context = example['context']
print(f"example[{index}]: question = '{question}'\ncontext = '{context}'")
print(f"expected answer = {example['answers']['text'][0]}")
actual_answer = answer_question(question, context, model, tokenizer)
print(f"actual answer = {actual_answer}")

example[25843]: question = 'what madonna single is credited as helping bring house music to the mainstream?'
context = 'The early 1990s additionally saw the rise in mainstream US popularity for house music. Pop recording artist Madonna's 1990 single "Vogue" became an international hit single and topped the US charts. The single is credited as helping to bring house music to the US mainstream.'
expected answer = Vogue
actual answer = " vogue "


When I run this example I get the following output:

```
example[25843]: question = 'what madonna single is credited as helping bring house music to the mainstream?'
context = 'The early 1990s additionally saw the rise in mainstream US popularity for house music. Pop recording artist Madonna's 1990 single "Vogue" became an international hit single and topped the US charts. The single is credited as helping to bring house music to the US mainstream.'
expected answer = Vogue
actual answer = " vogue "
```

If you see something different, then make sure you set the seed to 155.  If you did that and you still get a different answer, then don't worry about it.  There are probably differences across varied compute platforms that are difficult to account for.

The point here is that the expected answer "Vogue" and the actual answer " vogue " are pretty similar and perhaps we shouldn't be counting this as incorrect.  

Let's try this example with a different seed:

In [11]:
# with seed = 98
set_seed(98)
tokenizer = BertTokenizerFast.from_pretrained('prajjwal1/bert-mini')
model = BertForQuestionAnswering.from_pretrained('prajjwal1/bert-mini')
index = 25_843
example = squad['train'][index]
question = example['question']
context = example['context']
print(f"example[{index}]: question = '{question}'\ncontext = '{context}'")
print(f"expected answer = {example['answers']['text'][0]}")
actual_answer = answer_question(question, context, model, tokenizer)
print(f"actual answer = {actual_answer}")

Seed set to: 98


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at prajjwal1/bert-mini and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


example[25843]: question = 'what madonna single is credited as helping bring house music to the mainstream?'
context = 'The early 1990s additionally saw the rise in mainstream US popularity for house music. Pop recording artist Madonna's 1990 single "Vogue" became an international hit single and topped the US charts. The single is credited as helping to bring house music to the US mainstream.'
expected answer = Vogue
actual answer = 1990s additionally saw the rise in mainstream us popularity for house music. pop recording artist madonna ' s 1990 single " vogue " became an international hit single and topped the us charts. the single is credited as helping to


## Vogue Bad!

When I run this example with seed = 98 I see the following output:

```
example[25843]: question = 'what madonna single is credited as helping bring house music to the mainstream?'
context = 'The early 1990s additionally saw the rise in mainstream US popularity for house music. Pop recording artist Madonna's 1990 single "Vogue" became an international hit single and topped the US charts. The single is credited as helping to bring house music to the US mainstream.'
expected answer = Vogue
actual answer = 1990s additionally saw the rise in mainstream us popularity for house music . pop recording artist madonna ' s 1990 single " vogue " became an international hit single and topped the us charts . the single is credited as helping to
```

In this case the actual answer contains "vogue" but it isn't a very tight answer that meets the expectations of the labeled data.  

## Bad Answers

Looking at this example is illuminating and we can start assembling a set of criteria that would help us improve our evaluation approach beyond exact string matching.  In the next section we are going to define a method "accept_answer" that takes in the expected answer and the actual answer and determines if it should be counted as correct or not.  We can contemplate ways to relax the matching to ignore case, whitespace, punctuation, etc.  However, one observation that might occur to you by studying these two examples is that the answer is not a substring of the context.  This is kind of silly because the answer is derived by choosing a start token index and an end token index from the provided context.  Unfortunately, the "reverse tokenization" method convert_tokens_to_string does not know how to exactly reconstitute the selected tokens in a way that is consistent with the original string.  Fortunately, we can ask the tokenizer to keep track of the token offsets which keep track of the where each token occurs in terms of the original input string.  So, let's rewrite our answer_question method to make sure it returns a proper substring

## TODO 10 Points - Rewrite answer_question

Please rewrite the method answer_question as answer_question_substring so that the returned answer is a substring of the provided context string

In [12]:

def answer_question_substring(question, context, model, tokenizer):
    """
    This method should return an answer predicted by the model that is a proper substring of the provided context.
    This new implementation will look very much like the implementation of 'answer_question' above except that
    we need to ask the tokenizer to keep track of character offsets and then use those offsets to extract a substring
    of the context variable.

    Use the parameter value setting "return_offsets_mapping=True" when calling the tokenizer.  Then use the resulting
    offsets which you can retrieve with this: "inputs['offset_mapping']".

    Note: My first attempt to implement this threw an error that told me to use BertTokenizerFast so this is why we are using it!
    Note: After I updated the tokenizer, I got the following error:
              BertForQuestionAnswering.forward() got an unexpected keyword argument 'offset_mapping'
          I worked around this error by simply removing 'offset_mapping' from the inputs after assigning it to a local variable and
          before sending inputs to the model.
    Note: This method should not use tokenizer.convert_tokens_to_string
    Note: the logic for using start_index and end_index is a little different here.  You'll have to think carefully about how to get
          the end character index.
    """
    inputs = tokenizer(question, context, return_tensors='pt', truncation=True, max_length=512, return_offsets_mapping=True)

    # Save the offset maping to a local variable, then delete it out of the inputs dictionary
    offsets = inputs['offset_mapping']
    del inputs['offset_mapping']
    # print("Offsets: ", offsets)

    input_ids = inputs["input_ids"].tolist()[0]
    token_type_ids = inputs['token_type_ids']
    context_start_index = (token_type_ids == 1).nonzero(as_tuple=True)[1].min().item()

    with torch.no_grad():
        outputs = model(**inputs)
        start_scores = outputs.start_logits
        end_scores = outputs.end_logits

    start_index = torch.argmax(start_scores[0][context_start_index:]) + context_start_index
    end_index = torch.argmax(end_scores[0][start_index:]) + start_index

    # answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(input_ids[start_index:end_index]))

    # print(start_index, end_index)
    # print(offsets[0][start_index], offsets[0][end_index])

    # Directly substring the context rather then converting the indexes to strings
    answer = context[offsets[0][start_index][0]: offsets[0][end_index][0]]
    return answer

## Vogue Better

Ok - let's try our "vogue" example with the updated answer_question_substring method:

In [13]:
# with seed = 155
set_seed(155)
tokenizer = BertTokenizerFast.from_pretrained('prajjwal1/bert-mini')
model = BertForQuestionAnswering.from_pretrained('prajjwal1/bert-mini')
index = 25_843
example = squad['train'][index]
question = example['question']
context = example['context']
print(f"example[{index}]: question = '{question}'\ncontext = '{context}'")
print(f"expected answer = {example['answers']['text'][0]}")
actual_answer = answer_question_substring(question, context, model, tokenizer)
print(f"actual answer = {actual_answer}")

Seed set to: 155


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at prajjwal1/bert-mini and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


example[25843]: question = 'what madonna single is credited as helping bring house music to the mainstream?'
context = 'The early 1990s additionally saw the rise in mainstream US popularity for house music. Pop recording artist Madonna's 1990 single "Vogue" became an international hit single and topped the US charts. The single is credited as helping to bring house music to the US mainstream.'
expected answer = Vogue
actual answer = "Vogue" 


When I run this code I get the following output:

```
example[25843]: question = 'what madonna single is credited as helping bring house music to the mainstream?'
context = 'The early 1990s additionally saw the rise in mainstream US popularity for house music. Pop recording artist Madonna's 1990 single "Vogue" became an international hit single and topped the US charts. The single is credited as helping to bring house music to the US mainstream.'
expected answer = Vogue
actual answer = "Vogue"
```

Again, if you don't get this exact result - don't worry - it may not be possible to get the model to behave exactly the same on all platforms.

This is much better!  Now, we are getting <"Vogue"> instead of <" vogue ">.  But it still isn't an exact string match and so simple exact string matching is not going to suffice if we think <"Vogue"> should be a valid answer (and we do!)


## TODO 10 Points - Answer Matching

In this programming exercise we are going to write a match criteria that accounts for small discrepancies in answers from the expected answer that we'd like to ignore for the sake of evaluating the model.  We are going to write a method called 'accept_answer' that determines whether or not an answer returned from the model is acceptable.  This method should account for following:

 * case (i.e. upper-, lower-, and proper-casing)
 * punctuation (e.g. double quotes)
 * whitespace
 * allow the generated answer to have 50% extra material
   * we will disallow the generated answer to have less material, since that will often have the effect of making the answer not specific enough.
 * others?  If you have ideas about how to improve our evaluation, then please include them! (not required)


In [14]:
def accept_answer(expected_answer, actual_answer):
    """
    This method should return a boolean that determines whether or not to accept the answer returned by the model
    (actual_answer) given the expected answer provided by the training data (expected_answer). This method should
    account for the factors listed above and can be considered complete when all of the unit tests pass.

    A good strategy for implementing this method is to return True if matching conditions are met and then return
    False at the end of the method.

    Hint: regular expressions will be useful here!

    """
    # print("Expected: ", expected_answer, "Actual: ", actual_answer)
    # Make everything lowercase
    expected_answer = expected_answer.lower()
    actual_answer = actual_answer.lower()

    # Strip the punctuation, keeping only letters and numbers
    expected_answer = "".join(char for char in expected_answer if (char.isalpha() or char.isdigit()))
    actual_answer = "".join(char for char in actual_answer if (char.isalpha() or char.isdigit()))

    # Get the length after punctuation is removed (which seems to pass the test cases)
    len_e = len(expected_answer)
    len_a = len(actual_answer)

    # If the answer is an exact match, it's good!
    if actual_answer == expected_answer:
        return True
    # Otherwise, make sure the actual generated answer contains no more than 50% additional content
    if len_a <= int(len_e * 1.5):
        # Make sure the actual answer contains the expected answer
        if expected_answer in actual_answer:
            return True

    return False

In [15]:
def test_accept_answer(answer1, answer2, accept):
    assert accept == accept_answer(answer1, answer2)
    assert accept == accept_answer(answer2, answer1)

In [16]:
# UNIT TESTS - you can quit working on 'accept_answer' when all of the unit tests pass
# exact strings should match!
test_accept_answer("Vogue", "Vogue", True)
# strings that differ by case should match
test_accept_answer("Vogue", "VOGUE", True)
test_accept_answer("Vogue", "vogue", True)
test_accept_answer("VOGUE", "vogue", True)
# strings that differ by punctuation should match
test_accept_answer("vogue", "VOGUE!", True)
test_accept_answer("vogue", '"Vogue"', True)
test_accept_answer("Marlon Brando, Jimmy Dean, on the cover of a magazine", "marlon-brando!! JIMMY DEAN??! on the COVER of a magazine....", True)
# answers that have extra material on the front or end but are no more than 50% larger than the expected answer
# we don't use test_accept_answer because this criteria is not symmetrical
assert accept_answer("Jimmy Dean, on the cover of a magazine", "marlon-brando!! JIMMY DEAN??! on the COVER of a magazine....")
assert accept_answer("Year 2024", "The Year 2024")

# test some negative examples
test_accept_answer("vogue", "1990s additionally saw the rise in mainstream us popularity for house music. pop recording artist madonna ' s 1990 single \" vogue \" became an international hit single and topped the us charts. the single is credited as helping to", False)
test_accept_answer("vague", "vogue", False)
test_accept_answer("A", "B", False)

## Improved Evaluation

Now we are going to re-run our evaluation of SQuAD on BERT-mini.  Please run the following code to get a baseline performance of BERT-mini before finetuning with the updated evaluation code.  When I run this code I get 137 correct on the training data and 24 correct on the validation data.  Hopefully, you get a number that's better than what you got on Assignment 4.


In [45]:
def evaluate(examples, model, tokenizer):

    correct = 0
    progress = 0

    for example in examples:
        question = example['question']
        context = example['context']

        actual_answer = answer_question_substring(question, context, model, tokenizer)
        for expected_answer in example['answers']['text']:
            if accept_answer(expected_answer, actual_answer):
                correct += 1
                print(f"correct answer found: model answer = {actual_answer}, expected_answer = {expected_answer}")
                break
        progress += 1
        if progress % 5000 == 0:
            print(f"progress = {progress}, correct = {correct}")

    return correct


In [18]:
# let's start burning this data!  I get ~24 correct with the updated accept_answer method
evaluate(squad['validation'], model, tokenizer)

correct answer found: model answer = Sun Life Stadium, and the , expected_answer = Sun Life Stadium
correct answer found: model answer = ian Stewart, , expected_answer = Stewart
correct answer found: model answer = longitudinal waves, expected_answer = longitudinal waves
correct answer found: model answer = standardized cu, expected_answer = standardized
correct answer found: model answer = read lecture material , expected_answer = read lecture material
correct answer found: model answer = sin, expected_answer = sin
correct answer found: model answer = under a new multi-member proportional representation system. The State of , expected_answer = multi-member proportional representation system
correct answer found: model answer = d Protestantism in , expected_answer = Protestantism
correct answer found: model answer = E.I. du Pont, a , expected_answer = E.I. du Pont
correct answer found: model answer = Visa Inc, expected_answer = Visa Inc.
correct answer found: model answer = packet swit

27

# Finetuning BERT-mini

In this part of the homework we will finetune BERT-mini and see if we get better performance.  As I write this, I haven't yet verified that this will work - so, we shall see!

I had to install a package called 'accelerate' to get the following code to run.

In [19]:
pip install accelerate -U

In [20]:
# we will start by finetuning BERT-mini with a small subset of the SQuAD training data
squad_subset = squad['train'].select(range(1000))
print(squad_subset)

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 1000
})


In [21]:
# next let's preprocess the training data with the following code
# note that we have to manually add the start_positions and end_positions.  See the documentation here: https://huggingface.co/transformers/v3.0.2/model_doc/bert.html#bertforquestionanswering
def preprocess_function(examples):
    questions = [q.strip() for q in examples['question']]
    contexts = [c.strip() for c in examples['context']]
    inputs = tokenizer(questions, contexts, max_length=384, truncation=True, padding='max_length', return_tensors='pt')

    start_positions = []
    end_positions = []

    for i in range(len(examples['answers'])):
        answer = examples['answers'][i]['text'][0]
        start_idx = contexts[i].find(answer)
        end_idx = start_idx + len(answer)

        start_positions.append(start_idx)
        end_positions.append(end_idx)

    inputs.update({'start_positions': start_positions, 'end_positions': end_positions})
    return inputs

tokenized_squad = squad_subset.map(preprocess_function, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [22]:
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [23]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad,
    eval_dataset=tokenized_squad,
)

In [24]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch,Training Loss,Validation Loss
1,No log,5.913020
2,No log,5.880436
3,No log,5.868732


TrainOutput(global_step=189, training_loss=5.927229301008598, metrics={'train_runtime': 36.8429, 'train_samples_per_second': 81.427, 'train_steps_per_second': 5.13, 'total_flos': 21842376192000.0, 'train_loss': 5.927229301008598, 'epoch': 3.0})

## Training Problems!

When I run trainer.train I get the following ValueError:

```
You are trying to save a non contiguous tensor: `bert.encoder.layer.0.attention.self.query.weight` which is not allowed. It either means you are trying to save tensors which are reference of each other in which case it's recommended to save only the full tensors, and reslice at load time, or simply call `.contiguous()` on your tensor to pack it before saving.
```

Bing Copilot tells me to resolve this by overriding BertForQuestionAnswering's save method as follows:

In [25]:
class CustomBertForQuestionAnswering(BertForQuestionAnswering):
    def save_pretrained(self, save_directory, **kwargs):
        # Ensure all tensors are contiguous before saving
        for param in self.parameters():
            if not param.is_contiguous():
                param.data = param.data.contiguous()
        super().save_pretrained(save_directory, **kwargs)

In [26]:
model = CustomBertForQuestionAnswering.from_pretrained('prajjwal1/bert-mini')

Some weights of CustomBertForQuestionAnswering were not initialized from the model checkpoint at prajjwal1/bert-mini and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
#let's try again!
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad,
    eval_dataset=tokenized_squad,
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,5.902536
2,No log,5.870006
3,No log,5.859000


TrainOutput(global_step=189, training_loss=5.912035099413029, metrics={'train_runtime': 19.7801, 'train_samples_per_second': 151.668, 'train_steps_per_second': 9.555, 'total_flos': 21842376192000.0, 'train_loss': 5.912035099413029, 'epoch': 3.0})

Here's what I see when I finetune the model with this setup:

Epoch 	Training Loss 	Validation Loss
1 	No log 	5.895106
2 	No log 	5.864199
3 	No log 	5.853216

TrainOutput(global_step=189, training_loss=5.906726009631283, metrics={'train_runtime': 33.7988, 'train_samples_per_second': 88.76, 'train_steps_per_second': 5.592, 'total_flos': 21842376192000.0, 'train_loss': 5.906726009631283, 'epoch': 3.0})

What happened?  What do we do now?

Well - the validation loss went from 5.895106 to 5.853216 - so this suggests that training could be running in the right direction. The model should have saved to your notebook's current working directory in a subdirectory named "results".  Go look in the directory and make note of the name of the directory where the finetuned model was written to.  Let's load it and evaluate on the validation data:


In [28]:
model = BertForQuestionAnswering.from_pretrained('results/checkpoint-189')

## No more warning!

When you load the checkpoint model you should no longer see the warning about newly initialized weights - because now the model actually has them!  Woot!

In [29]:
evaluate(squad['validation'], model, tokenizer)

correct answer found: model answer = 20–18, by , expected_answer = 20–18
correct answer found: model answer = repeatedly burned out, , expected_answer = repeatedly burned out
correct answer found: model answer = "do not disturb" sign , expected_answer = "do not disturb" sign
correct answer found: model answer = directly via their adjacency matrices, , expected_answer = directly via their adjacency matrices
correct answer found: model answer = corrupt in its ways and had , expected_answer = corrupt in its ways
correct answer found: model answer = fourteen points , expected_answer = fourteen points
correct answer found: model answer = representatives elected to either house of parliament. , expected_answer = representatives elected to either house of parliament
correct answer found: model answer = twenty-five widows who settled in Dover, and there is no , expected_answer = twenty-five widows who settled in Dover
correct answer found: model answer = by having colloblasts, which , expected

26

When I rerun the evaluation on the validation data using the checkpoint model, it gets ~21 correct.  So, that wasn't very promising except for the fact that the machinery seems to be working.  Let's kick off a more substantial finetuning job next.

In [30]:
# let's prepare all of the training data
tokenized_squad = squad['train'].map(preprocess_function, batched=True)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

In [31]:
# let's restart with the original model
model = CustomBertForQuestionAnswering.from_pretrained('prajjwal1/bert-mini')

Some weights of CustomBertForQuestionAnswering were not initialized from the model checkpoint at prajjwal1/bert-mini and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad,
    eval_dataset=tokenized_squad,
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,5.494000,nan
2,5.338500,nan
3,5.281300,nan


TrainOutput(global_step=16425, training_loss=5.4380672980878995, metrics={'train_runtime': 1779.7653, 'train_samples_per_second': 147.658, 'train_steps_per_second': 9.229, 'total_flos': 1913370312043008.0, 'train_loss': 5.4380672980878995, 'epoch': 3.0})

When I train on all of the training data for three epochs it takes 50 minutes (go take a break!) and I see the following:

```
Epoch 	Training Loss 	Validation Loss
1 	5.836800 	5.698300
2 	5.530100 	5.395320
3 	5.434300 	5.319762
```

This is moving in the right direction!  Let's see how it evaluation on the validation comes out.  When I consult the 'results' directory in my notebooks workspace I see that the largest and final checkpoint is 16425 as shown next:

In [46]:
model = BertForQuestionAnswering.from_pretrained('results/checkpoint-16425')

In [34]:
evaluate(squad['validation'], model, tokenizer)

correct answer found: model answer = Levi's Stadium, expected_answer = Levi's Stadium
correct answer found: model answer = Levi's Stadium, expected_answer = Levi's Stadium.
correct answer found: model answer = 1874, , expected_answer = 1874
correct answer found: model answer = 1886 , expected_answer = 1886
correct answer found: model answer = = NP, expected_answer = NP
correct answer found: model answer = , expected_answer = .
correct answer found: model answer = states the Commission , expected_answer = the Commission
progress = 5000, correct = 7
correct answer found: model answer = 2002, , expected_answer = 2002
correct answer found: model answer = 1950s, , expected_answer = 1950s
correct answer found: model answer = under the terms of the Mental Health (Care and Treatment) (Scotland) Act 2003, expected_answer = Mental Health (Care and Treatment) (Scotland) Act 2003
correct answer found: model answer = ". British , expected_answer = British
correct answer found: model answer = , expe

14

When I run evaluation on the validation data I get ~18 correct. Bummer!  I was hoping performance would start getting better. So, I ran it for 10 more epochs (which took nearly 3 hours) and the "validation loss" went down to 4.013023 and the resulting model gave me ~52 correct which is trending in the right direction.  I quote "validation loss" because the astute observer will note that we are using the training data for both 'training loss' and 'validation loss'.  

## TODO - 20 Points - Finetuning

The last task for this assignment is to experiment with different approaches to finetuning BERT-mini for this task and see how good you can get performance to improve on the validation data.  You can try training for more epochs, try different learning rates, different values of weight decay, and different values for per_device_train_batch_size.  And we'll see if anyone can find a training strategy that actually works.  Please get advice for how best to accomplish this from any resource on the Internet that you wish.  If you get great results, then please share what you think the reason was.  If you get poor results, please do the same!  I would be curious to know if the evaluation on the training data improves too - it should (because it's cheating!).


### Experiment 1 - Increase the Learning Rate and Epochs

In [49]:
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [50]:
model = CustomBertForQuestionAnswering.from_pretrained('prajjwal1/bert-mini')

Some weights of CustomBertForQuestionAnswering were not initialized from the model checkpoint at prajjwal1/bert-mini and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [51]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad,
    eval_dataset=tokenized_squad,
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,5.628000,nan
2,5.463600,nan
3,5.377000,nan
4,5.304500,nan
5,5.252300,nan
6,5.180500,nan
7,5.192400,nan
8,5.181000,nan


TrainOutput(global_step=43800, training_loss=5.348922160945526, metrics={'train_runtime': 4709.0415, 'train_samples_per_second': 148.818, 'train_steps_per_second': 9.301, 'total_flos': 5102320832114688.0, 'train_loss': 5.348922160945526, 'epoch': 8.0})

In [52]:
evaluate(squad['validation'], model, tokenizer)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)

### Experiment 2 - Increase the Train Batch Size and Increase Weight Decay

In [38]:
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.05,
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [39]:
model = CustomBertForQuestionAnswering.from_pretrained('prajjwal1/bert-mini')

Some weights of CustomBertForQuestionAnswering were not initialized from the model checkpoint at prajjwal1/bert-mini and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [40]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad,
    eval_dataset=tokenized_squad,
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,5.588700,nan
2,5.458900,nan
3,5.408600,nan


TrainOutput(global_step=8214, training_loss=5.537080128058802, metrics={'train_runtime': 1713.6802, 'train_samples_per_second': 153.352, 'train_steps_per_second': 4.793, 'total_flos': 1913370312043008.0, 'train_loss': 5.537080128058802, 'epoch': 3.0})

In [44]:
evaluate(squad['validation'], model, tokenizer)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)

# Finetuning Experimental Results

Please detail in this markdown block the experiments you ran, the results you got, and what you learned.  

# PLEASE REPLACE WITH YOUR EXPERIMENTAL RESULTS HERE